# ANTARES Cloud Forge — Sovereign QLoRA Fine-Tuner

Optimized for **Google Colab Free Tier (Tesla T4 16GB VRAM)**.
- **Base Model**: `unsloth/DeepSeek-R1-Distill-Qwen-14B-bnb-4bit` (Default) or `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit`
- **Dataset**: 3,666 master sovereign pairs embedded inline (zero manual file uploads required)
- **Output**: Ready-to-use 4-bit/8-bit GGUF adapter exported directly to your computer

**How to run**: Click `Runtime` -> `Run all` (or press `Ctrl+F9`).

In [ ]:
# Step 1: Install Unsloth and Fast Acceleration Kernels (~45 seconds)
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# Step 2: Unpack Master Sovereign Training Dataset (3,666 Verified Pairs)
import os
import gzip
import base64

DATASET_FILE = "cloud_training_payload.jsonl"
if not os.path.exists(DATASET_FILE):
    print("Unpacking embedded sovereign dataset...")
    INLINE_DATASET_B64 = "..."
    with open(DATASET_FILE, "wb") as fp:
        fp.write(gzip.decompress(base64.b64decode(INLINE_DATASET_B64)))
    print(f"Dataset ready: {DATASET_FILE} ({os.path.getsize(DATASET_FILE):,} bytes)")
else:
    print(f"Using pre-existing {DATASET_FILE} ({os.path.getsize(DATASET_FILE):,} bytes)")

In [ ]:
# Step 3: Load 14B Target Model & Initialize Fast LoRA Projections
# Options: 'unsloth/DeepSeek-R1-Distill-Qwen-14B-bnb-4bit' or 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
TARGET_BASE_MODEL = "unsloth/DeepSeek-R1-Distill-Qwen-14B-bnb-4bit"

from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
print(f"Loading 4-bit base model: {TARGET_BASE_MODEL}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=TARGET_BASE_MODEL,
    max_seq_length=max_seq_length,
    dtype=None,  # Automatic detection
    load_in_4bit=True,
)

print("Attaching LoRA adapters to all attention and FFN linear layers...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("Model loaded and LoRA adapters armed!")

In [ ]:
# Step 4: Map Dataset & Execute Fine-Tuning Loop (~5-8 minutes on free Tesla T4)
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

print("Loading JSONL dataset...")
dataset = load_dataset("json", data_files=DATASET_FILE, split="train")

def format_prompts(batch):
    num_examples = len(batch[next(iter(batch))])
    texts = []
    has_chat_template = hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template is not None
    for i in range(num_examples):
        convo = batch.get('conversations', [None] * num_examples)[i]
        prompt = batch.get('prompt', [None] * num_examples)[i]
        chosen = batch.get('chosen', [None] * num_examples)[i]
        completion = batch.get('completion', [None] * num_examples)[i]
        msgs = batch.get('messages', [None] * num_examples)[i]
        
        if msgs and isinstance(msgs, list) and has_chat_template:
            texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
        elif convo and isinstance(convo, list):
            if has_chat_template:
                messages = []
                for turn in convo:
                    if isinstance(turn, dict):
                        role = 'user' if turn.get('from') in ('human', 'user') else 'assistant'
                        messages.append({'role': role, 'content': turn.get('value', '')})
                texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
            else:
                p, r = '', ''
                for turn in convo:
                    if isinstance(turn, dict):
                        if turn.get('from') in ('human', 'user'):
                            p = turn.get('value', '')
                        elif turn.get('from') in ('gpt', 'assistant'):
                            r = turn.get('value', '')
                texts.append(f'<|im_start|>user\n{p}<|im_end|>\n<|im_start|>assistant\n{r}<|im_end|>')
        elif prompt and chosen:
            if has_chat_template:
                messages = [{'role': 'user', 'content': prompt}, {'role': 'assistant', 'content': chosen}]
                texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
            else:
                texts.append(f'<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n{chosen}<|im_end|>')
        elif prompt and completion:
            if has_chat_template:
                messages = [{'role': 'user', 'content': prompt}, {'role': 'assistant', 'content': completion}]
                texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
            else:
                texts.append(f'<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n{completion}<|im_end|>')
        elif 'text' in batch and batch['text'][i]:
            texts.append(str(batch['text'][i]))
        else:
            texts.append('')
    return {'text': texts}

print("Mapping chat templates...")
dataset = dataset.map(format_prompts, batched=True, remove_columns=dataset.column_names)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=150,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=3407,
        output_dir='outputs',
        report_to='none',
    ),
)

print("Starting training loop...")
trainer_stats = trainer.train()
print(f"Training finished! Final loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# Step 5: Export GGUF Adapter and Auto-Download
import glob
from google.colab import files

output_dir = "antares_lora_gguf"
print("Exporting GGUF adapter (q8_0 quantization)...")
try:
    model.save_pretrained_gguf(output_dir, tokenizer, quantization_method="q8_0")
    ggufs = glob.glob(f"{output_dir}/*.gguf")
    if ggufs:
        target_file = ggufs[0]
        print(f"Downloading {target_file} to your computer...")
        files.download(target_file)
    else:
        print("GGUF folder empty, saving PEFT adapter...")
        model.save_pretrained("peft_adapter")
        !zip -r antares_adapter.zip peft_adapter/
        files.download("antares_adapter.zip")
except Exception as exc:
    print(f"Direct GGUF export note: {exc}. Saving PEFT adapter instead...")
    model.save_pretrained("peft_adapter")
    !zip -r antares_adapter.zip peft_adapter/
    files.download("antares_adapter.zip")

print("\n=== SUCCESS ===")
print("Once downloaded, drop the file into 'c:\\SYSUS\\models\\adapters\\' on Node 1.")